# 02 — Stage → masked dataframe

Turn `stage.csv` into the training table `masked_df.csv`. Steps:

1. round `start_x` / `start_y` / `duration` to ints
2. insert `kick_off` / `half_end` / `full_end` boundary sentinel rows per game/period
3. compute `time_elapsed` (seconds since the previous row in the same game/period)
4. integer-encode `event` and `result`; dump `event_masks.json` + `category_lookup.json`
5. attach the previous 3 events as lag columns
6. expand each event into 7 rows — one per field — with a causal mask over fields not yet revealed

In [27]:
import warnings
from statsbombpy.api_client import NoAuthWarning
from socceraction.data.statsbomb import StatsBombLoader
import pandas as pd
from statsbombpy import sb
import numpy as np
import json
import socceraction.spadl as spadl

warnings.filterwarnings("ignore", category=NoAuthWarning)
warnings.filterwarnings("ignore", message="credentials were not supplied")

pd.set_option("display.max_columns", None)  # Show all columns

In [28]:
df = pd.read_csv("data/stage.csv", index_col = 0)

In [30]:
df[["start_x", "start_y", 'duration']] = df[["start_x", "start_y", 'duration']].round(0).astype(int)

In [31]:
df['is_home'] = df['is_home'].astype(int)

## Boundary sentinels + `time_elapsed`

In [32]:
pieces = []
seen_games = set()

for (game_id, period_id), half in df.groupby(["game_id", "period_id"], sort=False):
    # Add kick_off sentinel at the start of each game's first period
    if game_id not in seen_games and period_id == 1:
        seen_games.add(game_id)
        
        kick_off_row = pd.DataFrame([[-2] * len(df.columns)], columns=df.columns)
        kick_off_row["game_id"] = game_id
        kick_off_row["period_id"] = period_id
        kick_off_row["index"] = 0  # Will be recalculated later
        kick_off_row["minute"] = 0
        kick_off_row["second"] = 0
        kick_off_row["type_name"] = 'kick_off'
        pieces.append(kick_off_row)
    
    # Append the actual half data
    pieces.append(half)
    
    # Add end_row sentinel
    last = half.iloc[-1]
    end_row = pd.DataFrame([[-2] * len(df.columns)], columns=df.columns)
    end_row["game_id"] = game_id
    end_row["period_id"] = period_id
    end_row["index"] = last["index"] 
    end_row["minute"] = last["minute"]
    end_row["second"] = last["second"]
    end_row["type_name"] = 'half_end' if period_id == 1 else 'full_end'
    pieces.append(end_row)

b_df = pd.concat(pieces, ignore_index=True)
b_df["index"] = b_df.groupby("game_id", sort=False).cumcount()

In [33]:
# Compute time_elapsed (seconds since previous row within the same game/period),
# alongside duration (kept, not replaced). Must run after sentinel rows (including kick_off) are in b_df,
# so period 1's first real event is diffed against kick_off's total_seconds == 0,
# and period 2's first real event (no prior row in its group) is filled with 0.
b_df['total_seconds'] = b_df['minute'] * 60 + b_df['second']
b_df['time_elapsed'] = b_df.groupby(['game_id', 'period_id'])['total_seconds'].diff().fillna(0)
b_df['time_elapsed'] = b_df['time_elapsed'].round(0).astype(int)
b_df = b_df.drop(columns=['total_seconds'])

In [34]:
b_df['time_elapsed'] = b_df['time_elapsed'].clip(upper=105)

In [35]:
b_df['type_name'].unique()

array(['kick_off', 'pass', 'dribble', 'take_on', 'foul', 'freekick_short',
       'bad_touch', 'tackle', 'corner_crossed', 'shot', 'clearance',
       'cross', 'throw_in', 'goalkick', 'keeper_save', 'freekick_crossed',
       'interception', 'keeper_claim', 'half_end', 'shot_freekick',
       'corner_short', 'full_end', 'shot_penalty', 'keeper_punch'],
      dtype=object)

## Integer-encode `event` / `result`, write masks + lookup

In [37]:
event_types = b_df['type_name'].unique()
e_lookup = {}

for i, event in enumerate(event_types):
    e_lookup[event] = i

b_df['event'] = b_df['type_name'].map(e_lookup)

result_types = [r for r in b_df["result_name"].unique() if r != -2]
r_lookup = {}

for i, result in enumerate(result_types):
    r_lookup[result] = i

b_df["result"] = b_df["result_name"].map(r_lookup)
b_df.loc[b_df["result_name"] == -2, "result"] = -2
b_df["result"] = b_df["result"].astype(int)

In [38]:
event_cols = ["is_home", "event", "start_x", "start_y", "time_elapsed", "duration", "result"]

event_masks = {col: sorted(b_df[col].unique().tolist()) for col in event_cols}

with open('data/event_masks.json', 'w') as f:
    json.dump(event_masks, f)

In [39]:
is_home_names = {-2: "boundary", 0: "away", 1: "home"}
event_names = {v: k for k, v in e_lookup.items()}
result_names = {-2: "boundary", **{v: k for k, v in r_lookup.items()}}

category_lookup = {
    "is_home": is_home_names,
    "event": event_names,
    "result": result_names,
}

with open('data/category_lookup.json', 'w') as f:
    json.dump(category_lookup, f)

In [41]:
b_df.describe()

,game_id,index,period_id,minute,second,duration,start_x,start_y,is_home,home_yellow_count,away_yellow_count,home_red_count,away_red_count,home_goal_count,away_goal_count,time_elapsed,event,result
count,7.416500e+05,741650.000000,741650.000000,741650.000000,741650.000000,741650.000000,741650.000000,741650.000000,741650.000000,741650.000000,741650.000000,741650.000000,741650.000000,741650.000000,741650.000000,741650.000000,741650.000000,741650.000000
mean,3.754162e+06,983.210354,1.491929,44.921796,29.292185,1.475672,53.578851,34.033543,0.511549,0.504768,0.611574,0.021106,0.022941,0.676169,0.531419,2.939442,3.009571,0.175687
std,1.098089e+02,576.874498,0.499935,27.044374,17.362852,1.432536,25.030190,20.192913,0.509009,0.828911,0.903947,0.183116,0.180801,0.976391,0.841457,6.762790,3.682224,0.409363
min,3.753972e+06,0.000000,1.000000,0.000000,0.000000,-2.000000,-2.000000,-2.000000,-2.000000,-2.000000,-2.000000,-2.000000,-2.000000,-2.000000,-2.000000,0.000000,0.000000,-2.000000
25%,3.754066e+06,487.000000,1.000000,21.000000,14.000000,1.000000,36.000000,17.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000
50%,3.754161e+06,975.000000,1.000000,45.000000,29.000000,1.000000,54.000000,34.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,2.000000,0.000000
75%,3.754257e+06,1463.000000,2.000000,68.000000,44.000000,2.000000,72.000000,52.000000,1.000000,1.000000,1.000000,0.000000,0.000000,1.000000,1.000000,2.000000,2.000000,0.000000
max,3.754351e+06,2399.000000,2.000000,101.000000,59.000000,68.000000,105.000000,68.000000,1.000000,6.000000,9.000000,2.000000,2.000000,6.000000,6.000000,105.000000,23.000000,5.000000


## Lag columns: the previous 3 events

In [42]:
def last_n_events(n, df):
    event_cols = ["is_home", "event", "start_x", "start_y", "time_elapsed", "duration", "result"]
    c = 0

    for c in range(n):
        c += 1
        event_cols_n = [col + '-' + str(c) for col in event_cols]

        df[event_cols_n] = (
            df.sort_values("index").groupby("game_id")[event_cols].shift(c, fill_value=-2)
        )
        
    return df

In [43]:
event_cols = ['is_home', 'event', 'start_x', 'start_y', 'time_elapsed', 'duration', 'result']
event_cols_1 = [col + '1' for col in event_cols]

In [45]:
df_3 = last_n_events(3, b_df)

## Expand each event into 7 masked rows

In [48]:
def mask_events(df, event_cols):
    n = len(event_cols)
    MASK = -1

    masked_df = df.loc[df.index.repeat(n)].reset_index(drop=True)
    masked_df['step'] = np.tile(np.arange(n), len(df))

    true_values = df[event_cols].loc[df.index.repeat(n)].reset_index(drop=True)
    masked_df['target'] = true_values.to_numpy()[np.arange(len(masked_df)), masked_df['step'].to_numpy()]

    for j, col in enumerate(event_cols):
        masked_df.loc[masked_df['step'] <= j, col] = MASK

    return masked_df

In [49]:
masked_df = mask_events(df_3, event_cols)

In [51]:
masked_df = masked_df[[
    'game_id', 'original_event_id', 'index',
    'is_home', 'event', 'type_name',
    'start_x', 'start_y',
    'time_elapsed', 'duration',
    'result', 'result_name',
    'period_id', 'minute', 'second',
    'home_goal_count', 'away_goal_count',
    'home_red_count', 'away_red_count',
    'home_yellow_count', 'away_yellow_count',
    'is_home-1', 'event-1', 'start_x-1', 'start_y-1', 'time_elapsed-1', 'duration-1', 'result-1',
    'is_home-2', 'event-2', 'start_x-2', 'start_y-2', 'time_elapsed-2', 'duration-2', 'result-2',
    'is_home-3', 'event-3', 'start_x-3', 'start_y-3', 'time_elapsed-3', 'duration-3', 'result-3',
    'step', 'target'
]]

In [53]:
masked_df.to_csv('data/masked_df.csv')